In [0]:
from pyspark.sql.functions import (
    col, expr, md5, concat_ws, lit, round, unix_timestamp, when, year, month
)

bronze_df = spark.table("test.bronze.yellow_taxi_trips")

silver_df = (
    bronze_df
    .filter(
        (col("tpep_pickup_datetime").isNotNull()) &
        (col("tpep_dropoff_datetime").isNotNull())  &
        (col("tpep_pickup_datetime") < col("tpep_dropoff_datetime")) &
        (col("trip_distance") > 0) &
        (col("passenger_count").between(0, 8)) &
        (col("total_amount") > 0) &
        (col("fare_amount") >= 0) &
        (col("tip_amount") >= 0) &
        (col("tolls_amount") >= 0) &
        (col("tpep_pickup_datetime") <= expr("current_timestamp()")) &
        (col("tpep_dropoff_datetime") <= expr("current_timestamp()")) &
        (col("trip_distance") <= 200) &
        (col("total_amount") <= 1000) &
        (col("fare_amount") <= 500) &
        (col("tip_amount") <= 200) &
        (col("tolls_amount") <= 100) &
        ((unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))).between(60, 28800)) &
        (year(col("tpep_pickup_datetime")).between(2009, 2025))
    )
    .withColumn(
        "trip_id",
        md5(
            concat_ws(
                "|",
                col("VendorID").cast("string"),
                col("tpep_pickup_datetime").cast("string"),
                col("tpep_dropoff_datetime").cast("string"),
                col("PULocationID").cast("string"),
                col("DOLocationID").cast("string"),
            )
        )
    )
    .withColumn("vendor_id", col("VendorID"))
    .withColumn(
        "vendor_name",
        when(col("VendorID") == 1, lit("Creative Mobile Technologies, LLC"))
        .when(col("VendorID") == 2, lit("Curb Mobility, LLC"))
        .when(col("VendorID") == 6, lit("Myle Technologies Inc"))
        .when(col("VendorID") == 7, lit("Helix"))
        .otherwise(lit("Unknown"))
    )
    .withColumn("pickup_datetime", col("tpep_pickup_datetime"))
    .withColumn("dropoff_datetime", col("tpep_dropoff_datetime"))
    .withColumn(
        "trip_duration_minutes",
        round(
            (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60, 2
        )
    )
    .withColumn("passenger_count", col("passenger_count"))
    .withColumn("trip_distance_km", round(col("trip_distance") * lit(1.60934), 2))
    .withColumn("ratecode_id", col("RatecodeID"))
    .withColumn(
        "ratecode_description",
        when(col("RatecodeID") == 1, lit("Standard rate"))
        .when(col("RatecodeID") == 2, lit("JFK"))
        .when(col("RatecodeID") == 3, lit("Newark"))
        .when(col("RatecodeID") == 4, lit("Nassau or Westchester"))
        .when(col("RatecodeID") == 5, lit("Negotiated fare"))
        .when(col("RatecodeID") == 6, lit("Group ride"))
        .when(col("RatecodeID") == 99, lit("Null/unknown"))
        .otherwise(lit("Other"))
    )
    .withColumn("store_and_fwd_flag", col("store_and_fwd_flag"))
    .withColumn(
        "store_and_fwd_description",
        when(col("store_and_fwd_flag") == "Y", lit("store and forward trip"))
        .when(col("store_and_fwd_flag") == "N", lit("not a store and forward trip"))
        .otherwise(lit("unknown"))
    )
    .withColumn("pickup_location_id", col("PULocationID"))
    .withColumn("dropoff_location_id", col("DOLocationID"))
    .withColumn("payment_type_id", col("payment_type"))
    .withColumn(
        "payment_type_description",
        when(col("payment_type") == 0, lit("Flex Fare trip"))
        .when(col("payment_type") == 1, lit("Credit card"))
        .when(col("payment_type") == 2, lit("Cash"))
        .when(col("payment_type") == 3, lit("No charge"))
        .when(col("payment_type") == 4, lit("Dispute"))
        .when(col("payment_type") == 5, lit("Unknown"))
        .when(col("payment_type") == 6, lit("Voided trip"))
        .otherwise(lit("Other"))
    )
    .withColumn("fare_amount", col("fare_amount"))
    .withColumn("extra", col("extra"))
    .withColumn("mta_tax", col("mta_tax"))
    .withColumn("tip_amount", col("tip_amount"))
    .withColumn("tolls_amount", col("tolls_amount"))
    .withColumn("improvement_surcharge", col("improvement_surcharge"))
    .withColumn("total_amount", col("total_amount"))
    .withColumn("congestion_surcharge", col("congestion_surcharge"))
    .withColumn("airport_fee", col("airport_fee"))
    .withColumn("pickup_year", year(col("tpep_pickup_datetime")))
    .withColumn("pickup_month", month(col("tpep_pickup_datetime")))
    .select(
        "trip_id",
        "vendor_id",
        "vendor_name",
        "pickup_datetime",
        "dropoff_datetime",
        "trip_duration_minutes",
        "passenger_count",
        "trip_distance_km",
        "ratecode_id",
        "ratecode_description",
        "store_and_fwd_flag",
        "store_and_fwd_description",
        "pickup_location_id",
        "dropoff_location_id",
        "payment_type_id",
        "payment_type_description",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "total_amount",
        "congestion_surcharge",
        "airport_fee",
        "pickup_year",
        "pickup_month"
    )
)

# Join to get location info
taxi_zones_df = spark.table("test.silver.taxi_zones")

silver_df = (
    silver_df
    .join(
        taxi_zones_df.withColumnRenamed("location_id", "pickup_location_id")
                     .withColumnRenamed("borough", "pickup_borough")
                     .withColumnRenamed("zone", "pickup_zone")
                     .withColumnRenamed("service_zone", "pickup_service_zone"),
        on="pickup_location_id",
        how="left"
    )
    .join(
        taxi_zones_df.withColumnRenamed("location_id", "dropoff_location_id")
                     .withColumnRenamed("borough", "dropoff_borough")
                     .withColumnRenamed("zone", "dropoff_zone")
                     .withColumnRenamed("service_zone", "dropoff_service_zone"),
        on="dropoff_location_id",
        how="left"
    )
)

(
    silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("pickup_year", "pickup_month") \
    .saveAsTable("test.silver.yellow_taxi_trips")
)

In [0]:
# %sql
## RUN THIS ONLY ONE THIME
# CREATE TABLE test.silver.taxi_zones AS
# SELECT LocationID as location_id,
# Borough as borough,
# Zone as zone,
# service_zone
#  FROM test.bronze.taxi_zones

In [0]:
%sql
SELECT * FROM test.silver.taxi_zones LIMIT 5 